In [ ]:
# 仓库根目录加入 sys.path（本仓库自包含，不依赖外部绝对路径）
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
from torch import nn
from d2l import torch as d2l
from deepseek_tokenizer import ds_token

import glob
import math
from torch.cuda.amp import GradScaler, autocast
from motex_utils.transformer import (AddNorm, PositionWiseFFN, transpose_output,
                                     transpose_qkv, DotProductAttention,
                                     sequence_mask, masked_softmax)


# V2版本
- 修复了V1的一些小问题，添加了注意力投影后的kv cache，减少了每次对投影的重复操作 

In [ ]:
class RopeMultiHeadAttentionKVCache(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens, num_heads, dropout, max_seq_len, bias=False, **kwargs):
        super(RopeMultiHeadAttentionKVCache, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.head_dim = num_hiddens // num_heads  # 重要修正
        self.attention = DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens, bias=bias)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=bias)
        self.cos, self.sin = self.precompute_rotary_emb(max_seq_len,  self.head_dim)

    def forward(self, queries, keys, values, valid_lens, state, i):
        queries = self.W_q(queries)
        keys = self.W_k(keys)
        values = self.W_v(values)

        # 先判断是否有缓存，以确定新 token 的绝对位置
        if state[0] is not None and state[0][i] is not None:
            cache_k, cache_v = state[0][i]
            query_offset = cache_k.shape[1]      # 历史长度即为新 token 的起始位置
        else:
            query_offset = 0

        # 转置并施加 RoPE（训练时 offset 为 0 不影响）
        queries = transpose_qkv(queries, self.num_heads)
        keys = transpose_qkv(keys, self.num_heads)
        values = transpose_qkv(values, self.num_heads)

        queries = self.apply_rotary_pos_emb(queries, self.cos, self.sin, offset=query_offset)
        keys = self.apply_rotary_pos_emb(keys, self.cos, self.sin, offset=query_offset)

        # 拼接历史缓存
        if state[0] is not None and state[0][i] is not None:
            keys = torch.cat((cache_k, keys), dim=1)
            values = torch.cat((cache_v, values), dim=1)

        # 更新缓存（推理时）
        if not self.training and state[0] is not None:
            state[0][i] = (keys, values)

        # 注意：valid_lens 的处理需移到多头复制前
        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens, repeats=self.num_heads, dim=0)

        output = self.attention(queries, keys, values, valid_lens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat), state
    
    def precompute_rotary_emb(self, max_seq_len, d, base=10000):
        theta = 1.0 / (base ** (torch.arange(0, d, 2, dtype=torch.float) / d))
        positions = torch.arange(max_seq_len, dtype=torch.float)
        angles = positions.unsqueeze(1) * theta.unsqueeze(0)  # (max_seq_len, d//2)
        cos = torch.cos(angles)
        sin = torch.sin(angles)
        return cos, sin

    def apply_rotary_pos_emb(self, x, cos, sin, offset=0):
        seq_len = x.shape[-2]
        d = x.shape[-1]
        half_d = d // 2

        # 确保 cos/sin 维度正确
        cos = cos[offset:offset+seq_len, :].to(x.device)  # (seq_len, half_d)
        sin = sin[offset:offset+seq_len, :].to(x.device)
        while cos.dim() < x.dim():
            cos = cos.unsqueeze(0)
            sin = sin.unsqueeze(0)

        x_left = x[..., :half_d]
        x_right = x[..., half_d:]
        x_rotated_left = x_left * cos - x_right * sin
        x_rotated_right = x_left * sin + x_right * cos
        return torch.cat([x_rotated_left, x_rotated_right], dim=-1)


In [ ]:
class GPTDecoderKVCacheBlock(nn.Module):
    def __init__(self, query_size, key_size,value_size,num_hiddens, norm_shape, ffn_num_input, ffn_num_hiddens,
                 num_heads, dropout, i, max_seq_len):
        super().__init__()
        self.i = i
        self.attention = RopeMultiHeadAttentionKVCache(
            key_size=key_size,
            query_size=query_size,
            value_size=value_size,
            num_hiddens=num_hiddens,
            num_heads=num_heads,
            dropout=dropout,
            max_seq_len=max_seq_len
        )
        self.addnorm1 = AddNorm(norm_shape, dropout)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, num_hiddens)
        self.addnorm2 = AddNorm(norm_shape, dropout)

    def forward(self, X, state=None, valid_lens=None):
        """
        X: (batch, seq_len, num_hiddens)
        state: 外部传入的缓存列表，结构为 [ [None]*num_layers, ... ]，
               训练时传 None；推理时 state[0] 必须为长度为 num_layers 的列表。
        valid_lens: 因果掩码，推理时通常传 None
        """
        # 直接交给注意力，不再在 block 层做任何初始化
        attn_output, state = self.attention(X, X, X, valid_lens, state, self.i)

        Y = self.addnorm1(X, attn_output)
        Z = self.addnorm2(Y, self.ffn(Y))
        return Z, state

In [ ]:
class GPTDecoder(nn.Module):
    def __init__(self, query_size, key_size, value_size, 
                 num_layers,num_hiddens, num_heads,norm_shape, 
                 vocabs_size, ffn_num_input, ffn_num_hiddens, dropout,max_seq_len):
        super().__init__()
        self.token_embedding = nn.Embedding(vocabs_size, num_hiddens)
        self.blks = nn.Sequential()
        self.num_layers = num_layers
        
        for i in range(num_layers):
            self.blks.add_module(
                f"{i}", GPTDecoderKVCacheBlock(query_size=query_size, key_size=key_size, 
                                     value_size=value_size, num_hiddens=num_hiddens,
                                     norm_shape=norm_shape, ffn_num_input=ffn_num_input,
                                    ffn_num_hiddens=ffn_num_hiddens,num_heads=num_heads, dropout=dropout,max_seq_len=max_seq_len, i=i)
            )
   
    def forward(self, tokens, valid_lens, state=None):
        X = self.token_embedding(tokens)
        # 初始化缓存列表：训练/评估常以 state=None 调用（与 GPT_v1 一致）
        if state is None:
            state = [None] * self.num_layers
        elif state[0] is None:
            state[0] = [None] * self.num_layers
        for blk in self.blks:
            X, state = blk(X, state, valid_lens)
        return X, state
        

In [ ]:
class GPTModel(nn.Module):
    def __init__(self,query_size, key_size, value_size, 
                 num_layers,num_hiddens, num_heads,norm_shape, 
                 vocabs_size, ffn_num_input, ffn_num_hiddens, dropout,max_seq_len):
        super().__init__()
        self.dense = nn.Linear(num_hiddens, vocabs_size)
        self.decoder = GPTDecoder(query_size, key_size, value_size, 
                 num_layers,num_hiddens, num_heads,norm_shape, 
                 vocabs_size, ffn_num_input, ffn_num_hiddens, dropout,max_seq_len)
    
    def forward(self, X ,valid_lens, state=None):
        X, state = self.decoder(X, valid_lens,state)
        return self.dense(X), state

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)
    if isinstance(m, nn.Embedding):
        torch.nn.init.normal_(m.weight, mean=0, std=0.02)

In [ ]:
def evaluate_gpt(net, test_iter, device):
    """在测试集上计算准确率"""
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for tokens, labels, valid_lens in test_iter:
            tokens = tokens.to(device)
            labels = labels.to(device)
            valid_lens = valid_lens.to(device)
            logits, _ = net(tokens, valid_lens, None)
            preds = logits.argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.numel()
    net.train()
    return correct / total if total > 0 else 0

In [ ]:



def train_gpt_ckpt(net, loss, train_iter, test_iter, vocab_size, devices, num_steps,
                   lr=1e-4, warmup_steps=100, weight_decay=0.01,
                   ckpt_dir='./checkpoints', save_every=100, max_keep=50,
                   resume_from=None):
    """
    训练 GPT 模型（因果语言模型）
    支持混合精度、checkpoint 保存 / 恢复，并绘制 train loss, train acc, test acc
    """
    os.makedirs(ckpt_dir, exist_ok=True)

    net.apply(init_weights)
    net = net.to(devices)

    # ---------- 优化器、调度器、Scaler ----------
    optimizer = torch.optim.AdamW(
        net.parameters(),
        lr=lr,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=weight_decay
    )

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        return max(0.0, float(num_steps - step) / float(max(1, num_steps - warmup_steps)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = GradScaler()

    # ---------- 恢复检查点 ----------
    start_step = 0
    best_loss = float('inf')
    best_model_path = os.path.join(ckpt_dir, 'best_model.pth')

    if resume_from is not None and os.path.isfile(resume_from):
        
        checkpoint = torch.load(resume_from, map_location=devices)
        net.load_state_dict(checkpoint['model_state_dict'])
        
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        if 'scaler_state_dict' in checkpoint:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])
            
        start_step = checkpoint['step']
        best_loss = checkpoint.get('best_loss', float('inf'))
        total_loss = checkpoint.get('total_loss', 0.0)
        
        total_tokens = checkpoint.get('total_tokens', 0)
        print(f"从 {resume_from} 恢复，继续从 step {start_step} 训练")
    else:
        total_loss, total_tokens = 0.0, 0
        # print("从头开始训练")

    # ---------- 绘图 ----------
    animator = d2l.Animator(
        xlabel='step', ylabel='loss/acc',
        xlim=[1, num_steps],
        legend=['train loss', 'train acc', 'test acc']
    )

    timer, step = d2l.Timer(), start_step
    train_acc_sum, train_acc_count = 0.0, 0
    test_acc_cache = None

    net.train()
    while step < num_steps:
        for batch in train_iter:
            tokens, labels, valid_lens = batch
            tokens = tokens.to(devices)
            labels = labels.to(devices)
            valid_lens = valid_lens.to(devices)

            optimizer.zero_grad()
            timer.start()

            # 混合精度前向（torch.amp.autocast 才支持 device_type=）
            with torch.amp.autocast(device_type=devices):
                logits, _ = net(tokens, valid_lens, None)
                l = loss(logits.reshape(-1, vocab_size), labels.reshape(-1))

            scaler.scale(l).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            timer.stop()

            # 计算准确率
            preds = logits.argmax(dim=-1)
            correct = (preds == labels).sum().item()
            total = labels.numel()
            batch_acc = correct / total

            batch_loss = l.item()
            batch_tokens = total
            total_loss += batch_loss * batch_tokens
            total_tokens += batch_tokens
            train_acc_sum += correct
            train_acc_count += total

            step += 1

            # 保存 checkpoint
            if step % save_every == 0:
                ckpt_name = f'model_step_{step:06d}.pth'
                ckpt_path = os.path.join(ckpt_dir, ckpt_name)
                torch.save({
                    'model_state_dict': net.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'scaler_state_dict': scaler.state_dict(),
                    'step': step,
                    'best_loss': best_loss,
                    'total_loss': total_loss,
                    'total_tokens': total_tokens,
                }, ckpt_path)

                # 管理版本数量
                all_ckpts = sorted(glob.glob(os.path.join(ckpt_dir, 'model_step_*.pth')))
                if len(all_ckpts) > max_keep:
                    os.remove(all_ckpts[0])

            # 评估测试集
            if test_iter is not None and step % 100 == 0:
                test_acc = evaluate_gpt(net, test_iter, devices)
                test_acc_cache = test_acc

            # 计算平均指标
            avg_loss = total_loss / total_tokens if total_tokens > 0 else 0
            avg_train_acc = train_acc_sum / train_acc_count if train_acc_count > 0 else 0

            # 更新最佳模型
            if avg_loss < best_loss:
                best_loss = avg_loss
                if step % (save_every / 10) == 0:
                    torch.save(net.state_dict(), best_model_path)

            # 更新动画
            if step % 10 == 0 or step == num_steps:
                animator.add(step, (avg_loss, avg_train_acc,
                                    test_acc_cache if test_acc_cache is not None else 0.0))

            if step >= num_steps:
                break

    avg_loss = total_loss / total_tokens if total_tokens > 0 else 0
    avg_train_acc = train_acc_sum / train_acc_count if train_acc_count > 0 else 0
    print(f'训练完成！平均损失 = {avg_loss:.4f}, 平均训练准确率 = {avg_train_acc:.4f}')
    print(f'训练速度: {total_tokens / timer.sum():.1f} tokens/sec on {str(devices)}')
    print(f'最佳模型保存在: {best_model_path}')

In [ ]:
def predict_gpt(net, prompt, max_new_tokens, device,
                bos_token_id=None, eos_token_id=None):
    

    net.eval()
    prompt_ids = ds_token.encode(prompt)
    if len(prompt_ids) + max_new_tokens > net.decoder.blks[0].attention.cos.shape[0]:
        raise ValueError("序列长度将超出预计算的 RoPE 表，请增加 max_seq_len 或减小生成长度")
    # 1. 构造初始输入（包含 BOS 和 prompt）
    if bos_token_id is not None:
        # 改动1：兼容 bos_token_id 可能是列表或整数
        bos_id = bos_token_id[0] if isinstance(bos_token_id, (list, tuple)) else bos_token_id
        input_ids = torch.tensor([[bos_id] + prompt_ids], device=device)
    else:
        input_ids = torch.tensor([prompt_ids], device=device)

    num_layers = net.decoder.num_layers
    state = [[None] * num_layers]
    generated_ids = []

    # 改动2：提前提取 eos_id，避免在循环内多次索引
    eos_id = None
    if eos_token_id is not None:
        eos_id = eos_token_id[0] if isinstance(eos_token_id, (list, tuple)) else eos_token_id

    with torch.no_grad():
        # 预填充
        logits, state = net(input_ids, valid_lens=None, state=state)
        next_token_id = logits[0, -1, :].argmax(dim=-1).item()

        # 自回归循环
        for _ in range(max_new_tokens):
            # 改动3：使用提前提取的 eos_id，且增加了 None 检查（避免 eos_token_id 为 None 时报错）
            if eos_id is not None and next_token_id == eos_id:
                break

            generated_ids.append(next_token_id)

            input_ids = torch.tensor([[next_token_id]], device=device)
            logits, state = net(input_ids, valid_lens=None, state=state)
            next_token_id = logits[0, -1, :].argmax(dim=-1).item()

    output_ids = prompt_ids + generated_ids
    return ds_token.decode(output_ids, skip_special_tokens=True)

In [ ]:
devices = d2l.try_gpu()
loss = nn.CrossEntropyLoss()
batch_size, max_len, lr = 16, 256, 1e-4
num_steps = 20480

In [ ]:
net = GPTModel(vocabs_size = ds_token.vocab_size, num_hiddens=256, norm_shape=[256],
                    ffn_num_input=256, ffn_num_hiddens=256, num_heads=8,
                    num_layers=12, dropout=0.2, key_size=256, query_size=256,
                    value_size=256, max_seq_len=max_len)


In [ ]:
# ============================================================
# 数据加载（placeholder）
# 数据集与数据加载代码未随本仓库分发，请自行准备数据并在此接入，
# 例如：train_iter, test_iter = my_dataloader(batch_size, max_len)
# ============================================================
train_iter = test_iter = None  # TODO: 替换为你的数据接口后再运行
# train_gpt(net,train_iter, loss, ds_token.vocab_size, devices, num_steps, lr)
train_gpt_ckpt(net, loss, train_iter,train_iter,  ds_token.vocab_size, devices, num_steps, lr, ckpt_dir='./checkpoints/GPT-v2')

In [ ]:
output = predict_gpt(net, "你是谁", 128, devices, bos_token_id=ds_token.encode("<｜end▁of▁sentence｜>"), eos_token_id=ds_token.encode('<｜end▁of▁sentence｜>'))
print(output)